# RAG Module · Day 19 — **Advanced Prompt Engineering**

**Phase 1 · Module 3: Prompt Engineering & RAG Architecture** · ~2.5 hrs

The model is fixed; the **prompt** is your control surface. Today covers the techniques Anthropic's guide ranks by impact, in the order you should reach for them:

1. **Zero-shot** — a clear instruction, nothing else.
2. **Few-shot** — teach the format/behaviour with examples.
3. **Chain-of-thought (CoT)** — make the model reason *before* answering.
4. **Tree-of-thought (ToT)** — explore several reasoning paths, then pick the best.
5. **Structured output with XML tags** — Claude's native way to return machine-readable fields.
6. **Prompt version control** — a YAML store so prompts are reviewed and rolled back like code.

> **Runnable keyless:** a deterministic `FakeClaude` stands in for a real Bedrock Claude. It keys off the *prompt technique* to mimic how a real model's answer improves as the prompt gets better — same `.invoke(str) -> str` surface, so swapping in `ChatBedrock` is one line.

## 0. A keyless stand-in that *responds to technique*

The point of the notebook is the **prompt**, not the model. `FakeClaude` reacts to the cues each technique adds — examples, a "think step by step" instruction, XML tags — so you can *see* the output change. It's deterministic and clearly a stand-in; a real Claude reacts to the same cues, more fluently.

In [1]:
import re, textwrap

class FakeClaude:
    """Deterministic stand-in for Bedrock Claude. Reacts to prompt technique, not content."""
    KEYWORDS = {"SALARY": ["salary", "payroll", "wages"],
                "DINING": ["restaurant", "cafe", "coffee", "pizza", "dinner"],
                "TRANSFER": ["transfer", "sent to", "payment to"]}

    def invoke(self, prompt: str) -> str:
        p = prompt.lower()
        # --- classification task: SALARY / DINING / TRANSFER ---
        if "categorise" in p or "category" in p:
            txt = prompt.split("Transaction:")[-1].split("Category:")[0].lower()
            cat = self._by_keyword(txt)
            if cat is None:                                       # no obvious keyword
                examples = re.findall(r"Transaction: (.+)\nCategory: (\w+)", prompt)
                if examples:                                      # few-shot: match nearest example
                    cat = self._nearest(txt, examples) or "UNKNOWN"
                else:                                             # zero-shot: give up
                    cat = "UNKNOWN"
            return f"<category>{cat}</category>" if "<category>" in prompt else cat
        # --- reasoning task: affordability ---
        if "step by step" in p or "<thinking>" in p:
            return textwrap.dedent("""\\
                <thinking>
                Monthly income is 4000. Existing outgoings are 1500, so 2500 is free.
                The new loan repayment is 900. 900 is below 2500, and total debt-to-income
                is (1500+900)/4000 = 60%, which EXCEEDS the 45% policy limit.
                </thinking>
                <answer>DECLINE - debt-to-income ratio 60% exceeds the 45% limit.</answer>""")
        if "affordab" in p:                                      # zero-shot, no reasoning asked
            return "APPROVE"                                     # confident but wrong
        return "OK"

    def _by_keyword(self, txt):
        for cat, kws in self.KEYWORDS.items():
            if any(k in txt for k in kws):
                return cat
        return None

    def _nearest(self, txt, examples):
        words = set(re.findall(r"[a-z]+", txt))
        best, best_overlap = None, 0
        for ex_txt, label in examples:
            overlap = len(words & set(re.findall(r"[a-z]+", ex_txt.lower())))
            if overlap > best_overlap:
                best, best_overlap = label.upper(), overlap
        return best

llm = FakeClaude()
print(llm.invoke("Categorise this transaction.\nTransaction: Monthly SALARY from Acme Ltd"))

SALARY


☝ `FakeClaude.invoke(prompt)` is the whole interface — same as `ChatBedrock().invoke(...).content`. Everything below changes only the **string** we pass in and watches the answer improve.

## 1. Zero-shot — a clear instruction

The baseline: state the task precisely and ask. Works when the task is common and unambiguous. Cost: nothing extra. Reach for this **first** — often it's enough.

In [2]:
def zero_shot(transaction):
    return f"Categorise this bank transaction into one word.\nTransaction: {transaction}"

for t in ["Monthly SALARY from Acme Ltd", "Dinner at Luigi's Pizza", "Standing order to Mr Khan"]:
    print(f"{t:38} -> {llm.invoke(zero_shot(t))}")

Monthly SALARY from Acme Ltd           -> SALARY
Dinner at Luigi's Pizza                -> DINING
Standing order to Mr Khan              -> UNKNOWN


☝ Clear and cheap. But the last one — a **standing order** — has no category keyword, and zero-shot returns `UNKNOWN` rather than guessing. That's where examples earn their keep.

## 2. Few-shot — teach with examples

Show the model a handful of input→output pairs. It infers the pattern (format, edge cases, tone) and applies it. The single highest-leverage technique for classification and formatting.

In [3]:
EXAMPLES = [
    ("Monthly SALARY from Acme Ltd",        "SALARY"),
    ("Coffee at Central Cafe",              "DINING"),
    ("Standing order to landlord Mr Khan",  "TRANSFER"),
]

def few_shot(transaction):
    shots = "\n".join(f"Transaction: {t}\nCategory: {c}" for t, c in EXAMPLES)
    return f"Categorise each bank transaction.\n{shots}\nTransaction: {transaction}\nCategory:"

hard = "Rent standing order to landlord"         # no category keyword -> zero-shot gives up
print("zero-shot:", llm.invoke(zero_shot(hard)))
print("few-shot :", llm.invoke(few_shot(hard)))

zero-shot: UNKNOWN
few-shot : TRANSFER


☝ The same ambiguous input zero-shot gave up on (`UNKNOWN`) now resolves to **TRANSFER** — few-shot matches it against the *landlord* example, inferring the pattern with no keyword to lean on. Few-shot is *teaching by demonstration*.

## 3. Chain-of-thought — reason before answering

For anything involving arithmetic, policy rules, or multi-step logic, instruct the model to **work through it step by step** before committing to an answer. The reasoning both improves accuracy and gives you an auditable trace — critical in a regulated setting.

In [4]:
case = "Income 4000/month, existing outgoings 1500/month, new loan repayment 900/month."

# zero-shot: jumps straight to a verdict
print("--- zero-shot ---")
print(llm.invoke(f"Is this loan affordable? {case}"))

# chain-of-thought: reason first, then decide
cot = f"""Decide if this loan is affordable under our 45% debt-to-income policy.
Think step by step, then give a verdict.
{case}"""
print("\n--- chain-of-thought ---")
print(llm.invoke(cot))

--- zero-shot ---
APPROVE

--- chain-of-thought ---
\
                <thinking>
                Monthly income is 4000. Existing outgoings are 1500, so 2500 is free.
                The new loan repayment is 900. 900 is below 2500, and total debt-to-income
                is (1500+900)/4000 = 60%, which EXCEEDS the 45% policy limit.
                </thinking>
                <answer>DECLINE - debt-to-income ratio 60% exceeds the 45% limit.</answer>


☝ Zero-shot says **APPROVE** — fast and *wrong*. Forced to reason, the model computes DTI = 60%, sees it breaches the 45% limit, and correctly **declines**. The `<thinking>` block is the audit trail a reviewer (or regulator) can inspect. You can strip it before showing the customer.

## 4. Tree-of-thought — explore, then pick the best

CoT follows **one** line of reasoning. Tree-of-thought generates **several** candidate approaches, evaluates each, and selects the strongest — useful when a problem has multiple valid framings (e.g. three ways to restructure a loan). Here we orchestrate it in code: ask for candidates, score them, keep the winner.

In [5]:
def tree_of_thought(question, n=3):
    # 1. generate n candidate approaches (deterministic stand-ins)
    candidates = [
        {"plan": "Extend term to 7 years", "monthly": 620, "total_interest": 4200},
        {"plan": "Reduce principal via deposit", "monthly": 710, "total_interest": 2600},
        {"plan": "Keep term, accept higher DTI", "monthly": 900, "total_interest": 3100},
    ][:n]
    # 2. evaluate each against the policy (score = affordable AND cheapest interest)
    for c in candidates:
        c["affordable"] = c["monthly"] <= 750           # fits the free-cashflow budget
        c["score"] = (c["affordable"], -c["total_interest"])
    # 3. pick the best-scoring branch
    best = max(candidates, key=lambda c: c["score"])
    return candidates, best

cands, best = tree_of_thought("How should we restructure this loan?")
for c in cands:
    print(f'{c["plan"]:32} monthly={c["monthly"]}  interest={c["total_interest"]}  affordable={c["affordable"]}')
print(f"\n-> chosen: {best['plan']}  (affordable and lowest interest)")

Extend term to 7 years           monthly=620  interest=4200  affordable=True
Reduce principal via deposit     monthly=710  interest=2600  affordable=True
Keep term, accept higher DTI     monthly=900  interest=3100  affordable=False

-> chosen: Reduce principal via deposit  (affordable and lowest interest)


☝ ToT = **generate → evaluate → select**, instead of trusting a single chain. Branch 2 wins: affordable *and* cheapest. In production each branch could be a separate model call (even parallel LangGraph nodes — Day 9) with the model scoring them; the pattern is the same.

## 5. Structured output with XML tags

Claude is trained to respect **XML tags**, so wrapping the requested fields in tags is the most reliable way to get parseable output from it (JSON works too — yesterday's parser — but tags are Claude's native idiom and easier to stream). Ask for tags, then extract with a simple regex.

In [6]:
def tagged_prompt(case):
    return f"""Assess this loan and respond in exactly this format:
<verdict>APPROVE or DECLINE</verdict>
<dti>the debt-to-income percentage</dti>
<reason>one sentence</reason>

Think step by step inside <thinking> tags first.
{case}"""

raw = llm.invoke(tagged_prompt(case))
print(raw, "\n")

def extract_tag(tag, text):
    m = re.search(fr"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return m.group(1).strip() if m else None

print("verdict:", extract_tag("answer", raw))
print("thinking (audit):", extract_tag("thinking", raw)[:60], "...")

\
                <thinking>
                Monthly income is 4000. Existing outgoings are 1500, so 2500 is free.
                The new loan repayment is 900. 900 is below 2500, and total debt-to-income
                is (1500+900)/4000 = 60%, which EXCEEDS the 45% policy limit.
                </thinking>
                <answer>DECLINE - debt-to-income ratio 60% exceeds the 45% limit.</answer> 

verdict: DECLINE - debt-to-income ratio 60% exceeds the 45% limit.
thinking (audit): Monthly income is 4000. Existing outgoings are 1500, so 2500 ...


☝ Tags give you named fields you can pull with one regex (`extract_tag`) — and they let you separate the **reasoning** (`<thinking>`) from the **answer** so you keep the audit trail but show the customer only the verdict. This is Anthropic's recommended pattern for structured output from Claude.

## 6. Prompt version control — a YAML store

Prompts are production assets: a wording change can move accuracy or safety. Store them in **YAML** with a version, load by name, and fill with yesterday's `format_map`. Now a prompt change is a reviewable diff you can roll back — not a string buried in code.

In [7]:
import yaml

PROMPT_YAML = """
affordability:
  version: 3
  updated: 2026-07-11
  template: |
    Decide if this loan is affordable under our {dti_limit}% debt-to-income policy.
    Think step by step inside <thinking> tags, then give a <answer>.
    {case}
categorise:
  version: 1
  template: |
    Categorise this bank transaction into one word.
    Transaction: {transaction}
"""

STORE = yaml.safe_load(PROMPT_YAML)

def render(name, **vars):
    entry = STORE[name]
    print(f"[using {name} v{entry['version']}]")
    return entry["template"].format_map(vars)

prompt = render("affordability", dti_limit=45, case=case)
print(llm.invoke(prompt))

[using affordability v3]
\
                <thinking>
                Monthly income is 4000. Existing outgoings are 1500, so 2500 is free.
                The new loan repayment is 900. 900 is below 2500, and total debt-to-income
                is (1500+900)/4000 = 60%, which EXCEEDS the 45% policy limit.
                </thinking>
                <answer>DECLINE - debt-to-income ratio 60% exceeds the 45% limit.</answer>


☝ The prompt now has a **version** and lives in a diffable file. Bump `version`, review the change, and if accuracy drops on your eval set (Day 18's harness) you roll back to the previous entry. This is `PromptTemplate`/LangSmith-Hub thinking in 20 lines — prompts treated as code.

## 7. Recap — the prompt toolkit, by leverage

| Technique | Use when | Cost |
|---|---|---|
| **Zero-shot** | common, unambiguous task | none — try first |
| **Few-shot** | need a format or edge-case behaviour taught | a few example tokens |
| **Chain-of-thought** | arithmetic / policy / multi-step logic | reasoning tokens; gives an audit trail |
| **Tree-of-thought** | several valid approaches to compare | N× calls — generate, score, select |
| **XML tags** | need parseable, field-separated output from Claude | trivial; Claude's native idiom |
| **YAML prompt store** | prompts are production assets | version + diff + rollback |

**Your 2.5 hrs today**
- [ ] Read Anthropic's prompt-engineering guide end to end.
- [ ] Turn a mislabelled zero-shot case correct with **few-shot** examples.
- [ ] Force a wrong zero-shot verdict right with **chain-of-thought**.
- [ ] Implement **tree-of-thought**: generate candidates, score, select.
- [ ] Get **XML-tagged** output and extract fields with regex; separate `<thinking>` from `<answer>`.
- [ ] Put two prompts in a **YAML store** with versions; render with `format_map`.

**Barclays lens:** in a regulated bank the *prompt* is where correctness and auditability are engineered — CoT produces the reasoning trace a reviewer needs, XML tags keep it machine-readable, and a versioned prompt store means every wording change is reviewed and reversible. Tomorrow: the retrieval half — RAG architecture patterns and chunking that decide *what* the prompt even sees.
